# M5 Base Training + Drift Calibration — Phase 2

Trains **one** robust base TFT forecaster and **one** base PPO pricer on the
base-fit window (2011–2012 minus the calibration tail), then characterises
*normal* error/profit on the held-out tail to set the drift thresholds.

- Forecasting trigger: windowed **MASE > μ + k·σ** (k=2.0; k=1.5/2.5 re-derivable).
- RL trigger: rolling **profit_index < 1** for 2 consecutive checks.

Outputs land in `outputs/drift/` (`checkpoints/base/`, `results/calibration.json`).

> Runs as-is even before the real M5 download: it falls back to a mini-M5 fixture.
> Set `USE_REAL_M5=True` + `SMOKE=False` once `data/processed_m5/` is populated.

## 0. Setup

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

PROJECT_DIR = str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd())
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import drift_pipeline.core_pipeline as cp
from drift_pipeline import base_training as bt
print('project:', PROJECT_DIR)
print('device :', cp.DEVICE)

## 1. Data — real M5, or auto mini-M5 fixture

In [ ]:
USE_REAL_M5 = True   # uses data/processed_m5/ if present; else builds a mini fixture

demand_csv = Path(cp.CONFIG['paths']['demand_csv'])
rl_csv     = Path(cp.CONFIG['paths']['rl_csv'])

if USE_REAL_M5 and demand_csv.exists() and rl_csv.exists():
    DEMAND, RL = str(demand_csv), str(rl_csv)
    print('Using REAL M5 at', demand_csv.parent)
else:
    print('Real M5 not found -> building mini-M5 fixture (validates the full flow).')
    from dataset_generator.m5.build_m5 import _make_mini_m5, build
    raw = Path(PROJECT_DIR)/'data'/'_m5_mini_raw'
    fix = Path(PROJECT_DIR)/'data'/'processed_m5_mini'
    paths = build(raw if (_make_mini_m5(raw, n_days=1850, n_items=6) or True) else raw, fix)
    DEMAND, RL = str(paths['forecast']), str(paths['rl'])

## 2. (Optional) smoke knobs — shrink for a fast end-to-end check

In [ ]:
SMOKE = True   # set False for the real base run
if SMOKE:
    cp.CONFIG['forecasting']['max_epochs'] = 2
    cp.CONFIG['forecasting']['hidden_size'] = 16
    cp.CONFIG['forecasting']['hidden_continuous_size'] = 8
    cp.CONFIG['rl']['total_timesteps_per_task'] = 2000
    cp.CONFIG['timeline']['calib_step_days'] = 7
    print('SMOKE config applied:', cp.CONFIG['forecasting']['max_epochs'], 'epochs')
else:
    print('FULL run:', cp.CONFIG['forecasting']['max_epochs'], 'epochs,',
          cp.CONFIG['rl']['total_timesteps_per_task'], 'PPO steps')

## 3. Prepare splits

In [ ]:
data = cp.prepare_drift_data(DEMAND, RL)
cp.print_timeline_summary(data)

## 4. Train base models + calibrate + save

In [ ]:
result = bt.run_base_training_and_calibration(data)
result['paths']

## 5. Calibration summary — the drift thresholds

In [ ]:
import json
calib = json.loads(Path(result['paths']['calibration']).read_text())
fc, rl = calib['forecasting'], calib['rl']

print('Forecasting  normal MASE : mu=%.4f  sigma=%.4f  (n=%d windows)'
      % (fc['mase_mu'], fc['mase_sigma'], fc['n_windows']))
print('Retrain when windowed MASE exceeds:')
for k, v in fc['thresholds'].items():
    star = '  <-- default' if k == ('k=%s' % cp.CONFIG['drift']['fc_k_sigma']) else ''
    print('   %-7s  MASE > %.3f%s' % (k, v, star))

print()
print('RL  normal profit/window : mu=%.2f  sigma=%.2f  (n=%d windows)'
      % (rl['ref_profit_mu'], rl['ref_profit_sigma'], rl['n_windows']))
print('RL trigger: rolling profit_index < %.2f for %d consecutive checks'
      % (calib['config']['rl_profit_floor'], calib['config']['rl_consecutive']))

---
**Next (Phase 3):** load `outputs/drift/checkpoints/base/` + `calibration.json`,
walk forward weekly through 2013–2015, log the raw windowed-error stream, and
fire retrains (ewc / replay / sdft) when drift clears the threshold.